# Evidently Tutorial

To install Evidently using the pip package manager, run:

```$ pip install evidently```


If you want to see reports inside a Jupyter notebook, you need to also install the Jupyter nbextension. After installing evidently, run the two following commands in the terminal from the Evidently directory.

To install jupyter nbextension, run:

```$ jupyter nbextension install --sys-prefix --symlink --overwrite --py evidently```

To enable it, run:

```$ jupyter nbextension enable evidently --py --sys-prefix```

That's it!

In [1]:
!pip install evidently==0.4.14 "numpy<2" "pydantic<2"

In [2]:
!jupyter nbextension install --sys-prefix --symlink --overwrite --py evidently

Installing /usr/local/lib/python3.12/dist-packages/evidently/nbextension/static -> evidently
Removing: /usr/share/jupyter/nbextensions/evidently
Symlinking: /usr/share/jupyter/nbextensions/evidently -> /usr/local/lib/python3.12/dist-packages/evidently/nbextension/static
- Validating: OK

    To initialize this nbextension in the browser every time the notebook (or other app) loads:
    
          jupyter nbextension enable evidently --py --sys-prefix
    


In [3]:
!jupyter nbextension enable evidently --py --sys-prefix

Enabling notebook extension evidently/extension...
      - Validating: OK


In [4]:
import evidently


In [5]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_california_housing

from evidently import ColumnMapping

from evidently.report import Report
from evidently.metrics.base_metric import generate_column_metrics
from evidently.metric_preset import DataDriftPreset, TargetDriftPreset, DataQualityPreset, RegressionPreset
from evidently.metrics import *

from evidently.test_suite import TestSuite
from evidently.tests.base_test import generate_column_tests
from evidently.test_preset import DataStabilityTestPreset, NoTargetPerformanceTestPreset
from evidently.tests import *

In [6]:
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

## Load Data

In [7]:
data = fetch_california_housing(as_frame=True)
housing_data = data.frame


In [8]:
housing_data.rename(columns={'MedHouseVal': 'target'}, inplace=True)
housing_data['prediction'] = housing_data['target'].values + np.random.normal(0, 5, housing_data.shape[0])

In [9]:
reference = housing_data.sample(n=5000, replace=False)
current = housing_data.sample(n=5000, replace=False)

## Report

In [ ]:
report = Report(metrics=[
    DataDriftPreset(),
])

report.run(reference_data=reference, current_data=current)
report

In [ ]:
report = Report(metrics=[
    ColumnSummaryMetric(column_name='AveRooms'),
    ColumnQuantileMetric(column_name='AveRooms', quantile=0.25),
    ColumnDriftMetric(column_name='AveRooms'),

])

report.run(reference_data=reference, current_data=current)
report

In [ ]:
report = Report(metrics=[
    generate_column_metrics(ColumnQuantileMetric, parameters={'quantile':0.25}, columns=['AveRooms', 'AveBedrms']),
])

report.run(reference_data=reference, current_data=current)
report

In [ ]:
report = Report(metrics=[
    ColumnSummaryMetric(column_name='AveRooms'),
    generate_column_metrics(ColumnQuantileMetric, parameters={'quantile':0.25}, columns='num'),
    DataDriftPreset()
])

report.run(reference_data=reference, current_data=current)
report

In [14]:
report.as_dict()

{'metrics': [{'metric': 'ColumnSummaryMetric',
   'result': {'column_name': 'AveRooms',
    'column_type': 'num',
    'reference_characteristics': {'number_of_rows': 5000,
     'count': 5000,
     'missing': 0,
     'missing_percentage': 0.0,
     'mean': 5.4,
     'std': 1.8,
     'min': 0.89,
     'p25': 4.44,
     'p50': 5.25,
     'p75': 6.06,
     'max': 34.84,
     'unique': 4888,
     'unique_percentage': 97.76,
     'infinite_count': 0,
     'infinite_percentage': 0.0,
     'most_common': 5.0,
     'most_common_percentage': 0.16},
    'current_characteristics': {'number_of_rows': 5000,
     'count': 5000,
     'missing': 0,
     'missing_percentage': 0.0,
     'mean': 5.43,
     'std': 2.84,
     'min': 1.38,
     'p25': 4.41,
     'p50': 5.22,
     'p75': 6.02,
     'max': 141.91,
     'unique': 4894,
     'unique_percentage': 97.88,
     'infinite_count': 0,
     'infinite_percentage': 0.0,
     'most_common': 4.0,
     'most_common_percentage': 0.16}}},
  {'metric': 'ColumnQ

In [15]:
report.json()

'{"version": "0.4.14", "metrics": [{"metric": "ColumnSummaryMetric", "result": {"column_name": "AveRooms", "column_type": "num", "reference_characteristics": {"number_of_rows": 5000, "count": 5000, "missing": 0, "missing_percentage": 0.0, "mean": 5.4, "std": 1.8, "min": 0.89, "p25": 4.44, "p50": 5.25, "p75": 6.06, "max": 34.84, "unique": 4888, "unique_percentage": 97.76, "infinite_count": 0, "infinite_percentage": 0.0, "most_common": 5.0, "most_common_percentage": 0.16}, "current_characteristics": {"number_of_rows": 5000, "count": 5000, "missing": 0, "missing_percentage": 0.0, "mean": 5.43, "std": 2.84, "min": 1.38, "p25": 4.41, "p50": 5.22, "p75": 6.02, "max": 141.91, "unique": 4894, "unique_percentage": 97.88, "infinite_count": 0, "infinite_percentage": 0.0, "most_common": 4.0, "most_common_percentage": 0.16}}}, {"metric": "ColumnQuantileMetric", "result": {"column_name": "AveBedrms", "column_type": "num", "quantile": 0.25, "current": {"value": 1.0076657906880344}, "reference": {"val

In [16]:
#save report as HTML
report.save_html('report.html')

In [17]:
#report.save_json('report.json')



```
# This is formatted as code
```

## Test Suite
Generate "unit-tests" for your model.

In [ ]:
tests = TestSuite(tests=[
    TestNumberOfColumnsWithMissingValues(),
    TestNumberOfRowsWithMissingValues(),
    TestNumberOfConstantColumns(),
    TestNumberOfDuplicatedRows(),
    TestNumberOfDuplicatedColumns(),
    TestColumnsType(),
    TestNumberOfDriftedColumns(),
])

tests.run(reference_data=reference, current_data=current)
tests

In [ ]:
suite = TestSuite(tests=[
    NoTargetPerformanceTestPreset(),
])

suite.run(reference_data=reference, current_data=current)
suite

In [ ]:
suite = TestSuite(tests=[
    TestColumnDrift('Population'),
    TestMeanInNSigmas('HouseAge'),
    NoTargetPerformanceTestPreset(columns=['AveRooms', 'AveBedrms', 'AveOccup'])
])

suite.run(reference_data=reference, current_data=current)
suite

In [ ]:
suite = TestSuite(tests=[
    TestColumnDrift('Population'),
    TestShareOfOutRangeValues('Population'),
    generate_column_tests(TestMeanInNSigmas, columns='num'),

])

suite.run(reference_data=reference, current_data=current)
suite

In [22]:
suite.as_dict()

{'tests': [{'name': 'Drift per Column',
   'description': 'The drift score for the feature **Population** is 0.021. The drift detection method is Wasserstein distance (normed). The drift detection threshold is 0.1.',
   'status': 'SUCCESS',
   'group': 'data_drift',
   'parameters': {'stattest': 'Wasserstein distance (normed)',
    'score': 0.021,
    'threshold': 0.1,
    'detected': False,
    'column_name': 'Population'}},
  {'name': 'Share of Out-of-Range Values',
   'description': 'The share of values out of range in the column **Population** is 0 (0 out of 5000).  The test threshold is eq=0 ± 1e-12.',
   'status': 'SUCCESS',
   'group': 'data_quality',
   'parameters': {'condition': {'eq': {'value': 0,
      'relative': 1e-06,
      'absolute': 1e-12}},
    'value': 0.0,
    'left': None,
    'right': None}},
  {'name': 'Mean Value Stability',
   'description': 'The mean value of the column **AveBedrms** is 1.1. The expected range is from 0.47 to 1.71',
   'status': 'SUCCESS',
  

In [23]:
suite.json()

'{"version": "0.4.14", "tests": [{"name": "Drift per Column", "description": "The drift score for the feature **Population** is 0.021. The drift detection method is Wasserstein distance (normed). The drift detection threshold is 0.1.", "status": "SUCCESS", "group": "data_drift", "parameters": {"stattest": "Wasserstein distance (normed)", "score": 0.021, "threshold": 0.1, "detected": false, "column_name": "Population"}}, {"name": "Share of Out-of-Range Values", "description": "The share of values out of range in the column **Population** is 0 (0 out of 5000).  The test threshold is eq=0 \\u00b1 1e-12.", "status": "SUCCESS", "group": "data_quality", "parameters": {"condition": {"eq": {"value": 0, "relative": 1e-06, "absolute": 1e-12}}, "value": 0.0, "left": null, "right": null}}, {"name": "Mean Value Stability", "description": "The mean value of the column **AveBedrms** is 1.1. The expected range is from 0.47 to 1.71", "status": "SUCCESS", "group": "data_quality", "parameters": {"column_

In [24]:
#suite.save_html('test_suite.html')

In [25]:
#suite.save_json('test_suite.json')